In [ ]:
import os
os.chdir("..")

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib as mpl
import seaborn as sb
import numpy as np
import h5py
#from tqdm import tqdm
import json
from scipy.optimize import curve_fit, minimize
from scipy.stats import t as tDistribution
from scipy.stats import multivariate_t
from scipy.special import gamma as gammaFunc
from scipy.integrate import quad
from scipy.ndimage import gaussian_filter, gaussian_filter1d

In [ ]:
from hubbleflow.fittingfunctions import velocityCorrection, scaleFittingCurve, logSigmaFittingCurve, invdegFittingCurve, skewFittingCurve

with open("fitting_results.json", "r") as f:
    fitting_results = json.load(f)

fitting_curve = fitting_results["location"]["parameters"]
scale_parameters = fitting_results["scale"]["parameters"]
invdeg_parameters = fitting_results["inverse_degrees_of_freedom"]["parameters"]
skew_parameters = fitting_results["skewness"]["parameters"]

In [ ]:
datafolder = "data/"
imagefolder = "figures/"

In [ ]:
with h5py.File(datafolder+"UchuuLong_test.h5.preserve") as file:
	clusters = pd.DataFrame.from_records(file["clusters"][:], index="icl")
	galaxies = pd.DataFrame.from_records(file["galaxies"].fields(["x", "y", "z", "R", "r", "vz", "vR", "vTang", "upID", "icl"])[:])

In [ ]:
clusters

In [ ]:
galaxies

In [ ]:
R_bin_width = 0.5
R_bins = np.arange(0, galaxies["R"].max() + R_bin_width/2, R_bin_width)
galaxies["Rbinned"] = pd.cut(galaxies["R"], bins=R_bins, labels=R_bins[:-1] + R_bin_width/2)

In [ ]:
N_M_bins = 5
mass_bins = np.geomspace(clusters["Mvcl"].min(), clusters["Mvcl"].max(), N_M_bins + 1)
clusters["Mbinned"] = pd.cut(clusters["Mvcl"], bins=mass_bins, labels=np.sqrt(mass_bins[:-1] * mass_bins[1:]))
galaxies["Mbinned"] = clusters.loc[galaxies["icl"], "Mbinned"].to_numpy()

In [ ]:
fit_params = pd.read_csv("fit_params.csv", index_col=["Mbinned", "Rbinned"])
avged = fit_params.groupby("Rbinned", observed=False).mean()
fit_params

In [ ]:
from hubbleflow.fittingfunctions import PVcorrclass, PUclass, unnormalizedPosteriorClass
from hubbleflow.kerneldensityestimation import KernelDensityEstimation

H0 = 1 / 10

PVcorr = PVcorrclass(*fitting_curve)

PU_args = np.concatenate((scale_parameters, invdeg_parameters))
PU = PUclass(*PU_args)

Rhist, Rbins = np.histogram(galaxies["R"], bins=100, density=True)

binwidth = 0.01
bins = np.arange(-binwidth/2, 20+binwidth, binwidth)
PR = KernelDensityEstimation(galaxies["R"].to_numpy(), bins, bandwidth=binwidth/2*1.4, a=0, b=20)

unnormalizedPosterior = unnormalizedPosteriorClass(PR, PVcorr, PU, H0)

def normalizedPosterior(R, r, v, M):
	if np.isscalar(R):
		if R < r:
			return 0
		unnormalized_value = unnormalizedPosterior(R, r, v, M)
		normalization = quad(unnormalizedPosterior, r, 20, args=(r, v, M))[0]
		if normalization == 0:
			return 0
		return unnormalized_value / normalization
	value = unnormalizedPosterior(R, r, v, M)
	normalization = quad(unnormalizedPosterior, r, 20, args=(r, v, M))[0]
	if normalization == 0:
		return 0
	return value / normalization
def firstComponent(R, r, v):
	value = np.zeros_like(R)
	mask = R >= r
	value[mask] = r/R[mask]/np.sqrt(R[mask]**2-r**2)
	return value
def secondComponent(R, r, v, M):
	value = np.zeros_like(R)
	mask = R >= r
	value[mask] = PU(np.abs(v)-(PVcorr(R[mask]))*(np.cos(np.arcsin(r/R[mask]))), R[mask], M)
	return value

In [ ]:
#fig, axs = plt.subplots(2, 2, figsize=(10, 6))
fig, axs = plt.subplots(2, 2, figsize=(10, 4), sharex=True, layout="constrained")#, gridspec_kw={"wspace":0.05})
axs = axs.flatten()
color_normalizer = mpl.colors.LogNorm(vmin=fit_params.reset_index()["Mbinned"].min(), vmax=fit_params.reset_index()["Mbinned"].max())
palette = "plasma_r"
for i, col in enumerate(fit_params.columns):
	ax = axs[i]
	data = fit_params.reset_index()
	scplt = sb.scatterplot(data=data, x="Rbinned", y=col, hue="Mbinned", palette=palette, hue_norm=color_normalizer, legend=False, ax=ax)
	if col != "scale":
		ax.plot(avged.index, avged[col], color="dodgerblue", linestyle="--", lw=2, label="Average over Mass Bins")
	xs = np.linspace(avged.index.get_level_values("Rbinned").min(), avged.index.get_level_values("Rbinned").max(), 100)
	if col == "loc":
		ax.plot(xs, velocityCorrection(xs, *fitting_curve), color="r", label="Fitting model to average")
	elif col == "scale":
		for Mbin in fit_params.index.get_level_values("Mbinned").unique():
			ax.plot(xs, scaleFittingCurve(xs, Mbin, *scale_parameters), color="r")
		ax.plot([], [], color="r", label="Fitting mass-dependent model")
	elif col == "invdeg":
		ax.plot(xs, invdegFittingCurve(xs, *invdeg_parameters), color="r", label="Fitting model to average")
		ax.set_ylim(bottom=0)
	elif col == "skew":
		ax.plot(xs, skewFittingCurve(xs, *skew_parameters), color="r", label="Fitting model to average")
	ax.set_ylabel(col)
colormap = plt.cm.ScalarMappable(norm=color_normalizer, cmap=palette)
cbar = fig.colorbar(colormap, ax=axs, aspect=25, pad=0.02, label=r"$M \ [\mathrm{M}_\odot]$")
for ax, ylabel in zip(axs, [r"$\mu \ [\mathrm{v}_{200}]$", r"$\sigma(M) \ [\mathrm{v}_{200}]$", r"$\nu$", r"$\lambda$"]):
	ax.set_xlabel(r"$r \ [\mathrm{r}_{200}]$")
	ax.set_ylabel(ylabel)
	ax.set_xlim(0, 20)
	ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=5))
	ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=4))
	ax.grid(alpha=0.3)
	ax.legend()
fig.savefig(imagefolder+"fitted_parameters_overview.pdf")

In [ ]:
plt.close()

In [ ]:
def normOfT(x, loc, scale, invdeg, skew, nevals=101, randomsize=100000):
	samples = multivariate_t.rvs(size=randomsize, loc=np.zeros(3), shape=scale**2*np.eye(3), df=1/invdeg)
	radial_distances = np.linalg.norm(samples[:,:2], axis=1)
	left_edge = x[0] - 0.5 * (x[1] - x[0])
	right_edge = x[-1] + 0.5 * (x[-1] - x[-2])
	counts, bins = np.histogram(radial_distances, bins=np.linspace(left_edge,right_edge,nevals), density=True)
	bin_centers = 0.5 * (bins[1:] + bins[:-1])
	return np.interp(x, bin_centers, counts, left=0, right=0)

def skewedTDistPDF(x, loc, scale, invdeg, skew):
	mu, sig, lam, q = loc, scale, skew, 1/(2*invdeg)
	if q < 100:
		ratio_of_gammas = gammaFunc(q - 0.5) / gammaFunc(q)
	else:
		ratio_of_gammas = q**(-0.5)
	v = 1 / np.sqrt(q * (1/(2*q-2)*(1+3*lam**2) - 4*lam**2/np.pi*ratio_of_gammas**2))
	m = lam * v * sig * 2 * np.sqrt(q/np.pi) * ratio_of_gammas
	if q < 100:
		ratio_of_gammas = gammaFunc(0.5 + q) / gammaFunc(q)
	else:
		ratio_of_gammas = q**0.5
	val = ratio_of_gammas / v / sig / np.sqrt(np.pi*q) / (1 + (x-mu+m)**2 / (q*v**2*sig**2*(1+lam*np.where(x-mu+m< 0,-1,1))**2))**(0.5+q)
	if not np.isfinite(val).all():
		print(f"Fitting parameters: loc={loc}, scale={scale}, invdeg={invdeg}, skew={skew}", flush=True, end='; ')
		print(f"q={q}", flush=True)
	return val

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(5, 3), layout="constrained")

Rbin = 2.25
Mbin = np.sqrt(mass_bins[2]*mass_bins[3])
data = galaxies.loc[(galaxies["Rbinned"] == Rbin) & (galaxies["Mbinned"] == Mbin)]
parameters = fit_params.loc[(Mbin, Rbin)]
parameters_symmetrised = parameters.copy()
parameters_symmetrised["skew"] = 0

axs[0].hist(data["vR"], bins=np.linspace(-3,3,101), color="navy", density=True, alpha=0.5, label="Data")
xs = np.linspace(-3, 3, 1001)
axs[0].plot(xs, skewedTDistPDF(xs, *parameters), color="darkred", lw=3, label=r"Fitted $P\,(v_r; \mu, \sigma, \nu, \lambda)$")
axs[0].plot(xs, skewedTDistPDF(xs, *parameters_symmetrised), color="red", linestyle="--", label=r"Fitted $P\,(v_r; \mu, \sigma, \nu, 0)$")
axs[0].set_xlabel(r"$v_r \ [\mathrm{v}_{200}]$")
axs[0].set_xlim([-3, 3])

axs[1].hist(data["vTang"], bins=np.linspace(0,3,101), color="navy", density=True, alpha=0.5, label="Data")
xs = np.linspace(0, 3, 1001)
axs[1].plot(xs, normOfT(xs, *parameters), color="darkred", lw=3, label=r"Fitted $P\,(u_\Omega; \sigma, \nu)$")
axs[1].set_xlabel(r"$u_\Omega \ [\mathrm{v}_{200}]$")
axs[1].set_xlim([0, 3])

for ax in axs:
	ax.set_ylabel("PDF")
	ax.legend(loc="upper right")
	ax.spines[["top","right"]].set_visible(False)
	ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=4, prune='upper'))

fig.savefig(imagefolder+"fitted_velocity_distributions.pdf")

In [ ]:
plt.close()

In [ ]:
data_fit = fit_params.loc[(slice(None), slice(5, None)), :].copy()
data_fit = data_fit.groupby(level="Mbinned", observed=False).mean()
sigma_fit = curve_fit(logSigmaFittingCurve, data_fit.index.to_numpy(), np.log(data_fit["scale"]).to_numpy(), p0=[14, np.log(0.46)], bounds=([10, -np.inf], [20, np.inf]))[0]
sigma_fit

In [ ]:
fig, ax = plt.subplots(figsize=(5,2.2), layout="constrained")
Rmin = 5
data = fit_params.loc[(slice(None), slice(Rmin, None)), :].copy()
data = data.groupby(level="Mbinned", observed=False).mean()
ax.scatter(data.index, data["scale"], s=50, color="slateblue", label=r"Averaged data")
xs = np.linspace(clusters["Mvcl"].min(), clusters["Mvcl"].max(), 200)
ax.plot(xs, np.exp(logSigmaFittingCurve(xs, *sigma_fit)), color="red", label=r"Model")
ax.plot(xs, np.exp(-1/3*np.log(xs)+10), color="darkturquoise", ls="--", label=r"$\propto M^{-1/3}$ reference")
ax.legend(loc="lower left")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$M \ [\mathrm{M}_\odot]$")
ax.set_ylabel(r"$\kappa_\sigma \ [\mathrm{v}_{200}]$")
ax.spines[["top","right"]].set_visible(False)
fig.savefig(imagefolder+"fit_scale_parameter_dependency.pdf")

In [ ]:
plt.close()

In [ ]:
sliced = clusters.loc[clusters["Mbinned"] == clusters["Mbinned"].unique()[3]]
R_real = np.array([6, 12])
R_eval = R_real / sliced["Rvcl"].mean()
vr_at_R_eval = velocityCorrection(R_eval, *fitting_curve)
R_no_correction = vr_at_R_eval / (1/10)
R_no_correction_physical = R_no_correction * sliced["Rvcl"].mean()
print("Comparison of real distances vs distances measured without correction")
print(f"Along the line of sight (maximum distortion), for an average massive cluster of mass {sliced['Mvcl'].mean():.2e} M☉")
for i in range(len(R_real)):
	print(f"At real distance {R_real[i]:.2f} Mpc, the distance measured without correction is {R_no_correction_physical[i]:.2f} Mpc. A difference of {np.abs(R_no_correction_physical[i]-R_real[i]):.2f} Mpc. A relative error of {np.abs(R_no_correction_physical[i]-R_real[i])/R_real[i]*100:.2f}%.")

In [ ]:
Rmax = 10
R_bins = np.linspace(0, Rmax, 101)
R_bin_width = R_bins[1] - R_bins[0]
vR_bins = np.linspace(-2, 2, 101)
galaxies["Rbinned_plot"] = pd.cut(galaxies["R"], bins=R_bins, labels=R_bins[:-1] + R_bin_width/2)
data = galaxies#.loc[galaxies["upID"] == -1].copy()
data = data.loc[data["R"] <= Rmax].copy()
data["Rbinned_plot"] = pd.cut(data["R"], bins=R_bins, labels=R_bins[:-1] + R_bin_width/2)
values = []
for Rbin, group in data.groupby("Rbinned_plot", observed=False):
	if group.shape[0] < 3:
		counts = np.zeros_like(counts)
	else:
		counts, bins = np.histogram(group["vR"], bins=vR_bins, density=True)
	values.append(counts)
values = np.array(values)

fig, ax = plt.subplots(figsize=(5,2.5), layout="constrained")
RR, VV = np.meshgrid(0.5 * (R_bins[1:] + R_bins[:-1]), 0.5 * (vR_bins[1:] + vR_bins[:-1]), indexing="ij")
image = ax.pcolormesh(RR, VV, values, shading="auto", cmap="Purples")
fig.colorbar(image, ax=ax, label="Vertically normalised density")
ax.set_xlabel(r"$r \ [\mathrm{r}_{200}]$")
ax.set_ylabel(r"$v_r \ [\mathrm{v}_{200}]$")
xs = np.linspace(0, Rmax, 201)
ax.plot(xs, velocityCorrection(xs, *fitting_curve), color="red", label=r"Correction model")
ax.plot(xs, 1/10*xs, color="k", linestyle="--", label=r"Hubble Flow")
ax.legend(loc="lower right")
ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=5))
fig.savefig(imagefolder+"vR_vs_R_density.pdf")

In [ ]:
plt.close()

In [ ]:
new_clusters_ids = [788, 453, 478]
try:
	assert new_clusters_ids == clusters_ids
	recompute = False
except:
	clusters_ids = new_clusters_ids
	recompute = True
Rbins = np.linspace(0, 20, 51)
Rs = np.linspace(0, 20, 201)
delta_R = Rs[1] - Rs[0]
Rs = Rs[:-1] + delta_R/2
if recompute:
	posteriors = []

fig, axs = plt.subplots(3, 1, figsize=(5, 7), sharex=True, layout="constrained")
legend_kwargs = [
	{
		"Data" : {"label" : "True data"},
		"Baseline" : {"label" : "Baseline"},
		"Posterior" : {"label" : "Posterior"}
	}
, {}, {}]

for i, cluster_id in tqdm(enumerate(clusters_ids), total=len(clusters_ids)):
	# select data
	data = galaxies.loc[galaxies["icl"] == cluster_id]
	data = data.loc[data["R"] <= 19.8]
	ax = axs[i]
	# plot real data
	ax.hist(data["R"], bins=Rbins, density=True, color="navy", alpha=0.5, **legend_kwargs[i].get("Data", {}))
	# plot data reconstructed with Hubble flow
	z_hf = data["vz"] / H0
	R_hf = np.sqrt(data["r"]**2 + z_hf**2)
#	ax.hist(R_hf, bins=Rbins, density=True, alpha=0.5, **legend_kwargs[i].get("Baseline", {}))
	counts, bins = np.histogram(R_hf, bins=Rbins, density=True)
	bin_centers = 0.5 * (bins[1:] + bins[:-1])
	ax.plot(Rs, np.interp(Rs, bin_centers, counts), color="k", linestyle="--", **legend_kwargs[i].get("Baseline", {}))
	# plot posterior
	r, v, M = data[["r"]].to_numpy(), data[["vz"]].to_numpy(), clusters.loc[data["icl"], ["Mvcl"]].to_numpy()
	if recompute:
		post = unnormalizedPosterior(Rs, r, v, M)
		post = (post / (post.sum(axis=1) * delta_R)[:, np.newaxis]).mean(axis=0)
		posteriors.append(post)
	else:
		post = posteriors[i]
	ax.plot(Rs, post, color="red", **legend_kwargs[i].get("Posterior", {}))
	# annotate
	text = f"Cluster ID: {cluster_id}\nMass: {clusters.loc[cluster_id, 'Mvcl']:.2e} M☉\nN° galaxies: {data.shape[0]}"
	ax.text(0.05, 0.95, text, transform=ax.transAxes, va="top")
	ax.spines[['right', 'top']].set_visible(False)
	ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=4, prune='upper'))
	ax.set_ylabel("Galaxy number density")

axs[2].set_xlabel(r"$r \ [\mathrm{r}_{200}]$")
axs[2].set_xlim([0, 20])

fig.legend(loc="outside upper center")
fig.savefig(imagefolder+"posterior_distance_distributions.pdf")

In [ ]:
plt.close()

In [ ]:
r_edges = np.linspace(0, 10, 21)
r_evals = 0.5 * (r_edges[1:] + r_edges[:-1])
vz_edges = np.linspace(-2, 2, 13)
vz_evals = 0.5 * (vz_edges[1:] + vz_edges[:-1])
epsilon_r = (r_evals[1] - r_evals[0]) * 0.2
epsilon_vz = (vz_evals[1] - vz_evals[0]) * 0.2
print(f"r_evals: {r_evals}, epsilon_r: {epsilon_r}", flush=True)
print(f"vz_evals: {vz_evals}, epsilon_vz: {epsilon_vz}", flush=True)

In [ ]:
rr, vv = np.meshgrid(r_evals, vz_evals, indexing="ij")
how_many = np.empty_like(rr).flatten()
for i,(r,vz) in tqdm(enumerate(zip(rr.flatten(), vv.flatten())), total=rr.size):
	how_many[i] = galaxies.loc[((galaxies["r"] - r).abs() < epsilon_r) & ((galaxies["vz"] - vz).abs() < epsilon_vz)].shape[0]
how_many = how_many.reshape(rr.shape)
fig, ax = plt.subplots(figsize=(5,3), layout="constrained")
cbar = ax.pcolormesh(rr, vv, how_many, shading="auto", cmap="Purples")
fig.colorbar(cbar, label="Counts", aspect=30, pad=0.02)
for r, vz, val in zip(rr.flatten(), vv.flatten(), how_many.flatten()):
	ax.text(r, vz, f"{int(val)}", color="k", ha="center", va="center", fontsize=4)
ax.set_xlabel(r"$R \ [\mathrm{r}_{200}]$")
ax.set_ylabel(r"$v_z \ [\mathrm{v}_{200}]$")
ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=5))
fig.savefig(imagefolder+"r_vs_vz_counts.pdf")

In [ ]:
fig, axs = plt.subplots(vz_evals.shape[0], r_evals.shape[0], figsize=(3*r_evals.shape[0], 3*vz_evals.shape[0]), layout="constrained")
R_nbins = 21
error_baseline = np.empty((vz_evals.shape[0], r_evals.shape[0]))
error_model = np.empty((vz_evals.shape[0], r_evals.shape[0]))
for i, vz in tqdm(enumerate(vz_evals[::-1]), total=vz_evals.shape[0]):
	for j, r in tqdm(enumerate(r_evals), total=r_evals.shape[0]):
		ax = axs[i,j]
		data = galaxies.loc[((galaxies["r"] - r).abs() < epsilon_r) & ((galaxies["vz"] - vz).abs() < epsilon_vz)]
		counts, bins = np.histogram(data["R"], bins=R_nbins, density=True)
		baseline_zs = data["vz"] / H0
		baseline_Rs = np.sqrt(data["r"]**2 + baseline_zs**2)
		baseline_counts, _ = np.histogram(baseline_Rs, bins=bins, density=True)
		Rs = 0.5 * (bins[1:] + bins[:-1])
		delta_R = Rs[1] - Rs[0]
		rs, vzs, Ms = data[["r"]].to_numpy(), data[["vz"]].to_numpy(), clusters.loc[data["icl"], ["Mvcl"]].to_numpy()
		posteriors = unnormalizedPosterior(Rs, rs, vzs, Ms)
		posteriors = posteriors / (posteriors.sum(axis=1) * delta_R)[:, np.newaxis]
		values = posteriors.mean(axis=0)
		# plot
		text = "\n".join([rf"$r={r:.2f}$", rf"$v_{{z}}={vz:.2f}$", rf"$\mathrm{{count}}={data.shape[0]}$"])
		ax.text(0.5, 0.98, text, ha="center", va="top", transform=ax.transAxes)
		ax.hist(data["R"], bins=R_nbins, density=True, alpha=0.5, label="Data")
		ax.hist(baseline_Rs, bins=bins, density=True, alpha=0.5, label="Baseline")
		ax.plot(Rs, values, color="red", label="Posterior")
		if np.any(~np.isfinite(values)):
			print(f"Non-finite values encountered at r={r:.2f}, vz={vz:.2f}", flush=True)
		# compute error metric
		error_baseline[i,j] = np.sum(np.abs(baseline_counts - counts)) * delta_R
		error_model[i,j] = np.sum(np.abs(values - counts)) * delta_R
fig.savefig(imagefolder+"posterior_distance_distributions_grid.pdf")
plt.close()

In [ ]:
inspection_r = r_evals[[0, 1, 3, 9, 9, 14, 19]]
inspection_vz = vz_evals[[6, 9, 4, 8, 0, 7, 3]]

fig = plt.figure(figsize=(10, 6), layout="constrained")
subfigs = fig.subfigures(1, 2)


axs_color = subfigs[0].subplots(2, 1, sharex=True, sharey=True)

color_normaliser = mpl.colors.Normalize(vmin=0, vmax=error_model.max())
c0 = axs_color[0].pcolormesh(r_edges, vz_edges, error_model, shading="flat", norm=color_normaliser, cmap="Greens_r")
fig.colorbar(c0, ax=axs_color[0], label=r"$L^1$-distance posterior to data", aspect=20, shrink=0.85, pad=0.03)

color_normaliser = mpl.colors.LogNorm(vmin=1, vmax=100)
c1 = axs_color[1].pcolormesh(r_edges, vz_edges, error_baseline/error_model, shading="flat", norm=color_normaliser, cmap="RdYlBu")
fig.colorbar(c1, ax=axs_color[1], label=r"Improvement over baseline", aspect=20, shrink=0.85, pad=0.03)

for ax in axs_color:
	ax.set_ylabel(r"$v_z \ [\mathrm{v}_{200}]$")
	ax.yaxis.set_major_locator(plt.FixedLocator(np.linspace(-2, 2, 5)))
axs_color[1].set_xlabel(r"$R \ [\mathrm{r}_{200}]$")


axs_distr = subfigs[1].subplots(int(np.ceil(inspection_r.shape[0]/2)), 2)
axs_distr = axs_distr.flatten()

R_nbins = 51
def getLabel(i, which):
	if i >= 0:
		return None
	return which
for i, (r, vz) in enumerate(zip(inspection_r, inspection_vz)):
	ax = axs_distr[i]
	data = galaxies.loc[((galaxies["r"] - r).abs() < epsilon_r) & ((galaxies["vz"] - vz).abs() < epsilon_vz)]
	counts, bins = np.histogram(data["R"], bins=R_nbins, density=True)
	baseline_zs = data["vz"] / H0
	baseline_Rs = np.sqrt(data["r"]**2 + baseline_zs**2)
	baseline_counts, _ = np.histogram(baseline_Rs, bins=bins, density=True)
	Rs = 0.5 * (bins[1:] + bins[:-1])
	delta_R = Rs[1] - Rs[0]
	rs, vzs, Ms = data[["r"]].to_numpy(), data[["vz"]].to_numpy(), clusters.loc[data["icl"], ["Mvcl"]].to_numpy()
	posteriors = unnormalizedPosterior(Rs, rs, vzs, Ms)
	posteriors = posteriors / (posteriors.sum(axis=1) * delta_R)[:, np.newaxis]
	values = posteriors.mean(axis=0)
	# plot
	text = f"N° galaxies: {data.shape[0]}"
	ax.text(0.5, 0.7, text, ha="center", va="top", transform=ax.transAxes, fontsize=8)
	ax.hist(data["R"], bins=R_nbins, density=True, color="navy", alpha=0.5, label=getLabel(i, "Data"))
	ax.plot(Rs, np.interp(Rs, 0.5*(bins[1:]+bins[:-1]), baseline_counts), color="k", linestyle="--", label=getLabel(i, "Baseline"))
	ax.plot(Rs, values, color="red", label=getLabel(i, "Posterior"))
	ax.spines[['right', 'top']].set_visible(False)
#	ax.set_ylim(bottom=0, top=counts.max()*1.1)

for ax in axs_distr[:-1]:
	ax.set_xlabel(r"$r \ [\mathrm{r}_{200}]$")
	ax.set_ylabel("PDF")
	ax.set_yscale("log")
axs_distr[-1].hist([0], bins=np.linspace(-2,-1,3), density=True, color="navy", alpha=0.5, label="Data")
axs_distr[-1].plot([], [], color="k", linestyle="--", label="Baseline")
axs_distr[-1].plot([], [], color="red", label="Posterior")
axs_distr[-1].legend(loc="center", title="Probability densities:")
axs_distr[-1].set_xticks([])
axs_distr[-1].set_yticks([])
axs_distr[-1].spines[['right', 'top', 'left', 'bottom']].set_visible(False)


annotations = [f"({chr(ord('a') + i)})" for i in range(inspection_r.shape[0])]
axs_color[0].errorbar(inspection_r, inspection_vz, xerr=epsilon_r, yerr=epsilon_vz, linestyle="none", ecolor="white", capsize=3)
axs_color[1].errorbar(inspection_r, inspection_vz, xerr=epsilon_r, yerr=epsilon_vz, linestyle="none", ecolor="black", capsize=3)
for i, (r, vz) in enumerate(zip(inspection_r, inspection_vz)):
	axs_color[0].annotate(annotations[i], (r, vz), xytext=(r, vz+3*epsilon_vz), ha="center", va="center", color="white", fontsize=12)
	axs_color[1].annotate(annotations[i], (r, vz), xytext=(r, vz+3*epsilon_vz), ha="center", va="center", color="black", fontsize=12)
	axs_distr[i].annotate(annotations[i], (0.5, 0.95), xycoords="axes fraction", ha="center", va="top", color="k", fontsize=16)


fig.savefig(imagefolder+"error_models_comparison.pdf")

In [ ]:
plt.close()